In [ ]:
import sys, os

nb_path = "C:/Users/mgvbo/Documents/Simulations/Finesse3/"
sys.path.append(os.path.abspath(nb_path))

import finesse
import matplotlib.pyplot as plt
import time

from finesse.analysis.actions import (
    Xaxis,
    GetErrorSignals,
    CheckLinearity,
    RunLocks,
    DragLocks,
)
from finesse.knm import Map
from finesse.utilities.maps import circular_aperture
from finesse.plotting import plot_field

import numpy as np

# import pandas as pd
from tabulate import tabulate
import finesse.analysis.actions as fac
import scipy
from scipy.interpolate import make_splrep

import matplotlib

matplotlib.rcParams["figure.max_open_warning"] = 200

import helper_functions as hel

out_path = "C:/Users/mgvbo/Documents/Simulations/Finesse3/O5_baseline/ASC_Sensing/Output/Complete_Sensing/"

clr = [
    "tab:orange",
    "tab:pink",
    "tab:olive",
    "tab:purple",
    "tab:green",
    "tab:cyan",
    "tab:blue",
    "tab:red",
]

# %matplotlib widget

In [ ]:
def cart2pol(x, y):
    """Convert from Cartesian to polar coordinates.

    Example
    -------
    >>> theta, radius = pol2cart(x, y)
    """
    radius = np.hypot(x, y)
    theta = np.arctan2(y, x)
    return theta, radius


def compass(u, v, arrowprops=None):
    """
    Compass draws a graph that displays the vectors with
    components `u` and `v` as arrows from the origin.

    Examples
    --------
    >>> import numpy as np
    >>> u = [+0, +0.5, -0.50, -0.90]
    >>> v = [+1, +0.5, -0.45, +0.85]
    >>> compass(u, v)
    """

    angles, radii = cart2pol(u, v)
    fig, ax = plt.subplots(subplot_kw=dict(polar=True))

    kw = dict(arrowstyle="->")
    if arrowprops:
        kw.update(arrowprops)
    [
        ax.annotate("", xy=(angle, radius), xytext=(0, 0), arrowprops=kw)
        for angle, radius in zip(angles, radii)
    ]

    ax.set_ylim(0, np.max(radii))
    return fig, ax


def get_QNLS(model, axis=[10, 1000, 500]):
    kat = model.deepcopy()
    kat.parse(
        """
        # Differentially modulate the arm lengths
        fsig(1)
        sgen darmx sNarm.h
        sgen darmy sWarm.h phase=180

        # Output the full quantum noise limited sensitivity
        qnoised NSR_with_RP SRM1.p2.o nsr=True

        # Output just the shot noise limited sensitivity
        qshot NSR_without_RP SRM1.p2.o nsr=True
        """
    )

    return kat.run(f'xaxis(darmx.f, "log", {axis[0]}, {axis[1]}, {axis[2]})')


def dof_plot_rf(model, dof, detector, axis=[-1, 1, 1000], xscale=1, logy=True):
    kat = model.deepcopy()
    axis = np.array(axis, dtype=np.float64)
    axis[:2] *= xscale
    out = kat.run(fac.Xaxis(eval("kat." + dof + ".DC"), "lin", *axis, relative=True))
    fig, ax = plt.subplots(1, 1, figsize=(5, 3))

    ax.plot(out.x[0], out[detector + "_I"], label=detector + "_I", linestyle="-")
    ax.plot(out.x[0], out[detector + "_Q"], label=detector + "_Q", linestyle="--")
    ax.set_title(dof + " DC scan")
    ax.set_xlabel(dof + " DC")
    ax.set_ylabel(r"Amplitude $[\sqrt{W}]$")
    ax.legend(ncol=2)
    ax.grid(True)
    fig.set_tight_layout(True)
    return out


def get_dxdy(kat, bp, port, f="0"):
    # this function takes a kat file (should already contain a misalignment and be locked afterwards),
    # a beam tracing output (bp = some_kat.beam_trace()) and a port attached to a mirror where the de-
    # centering is computed.
    #
    # It attaches the necessary ADs to a temporary copy of the kat file and runs a 1-point simulation,
    # then returns the dx/dy decentering.

    if f not in ["0", "eom1.f", "eom2.f", "eom3.f"]:
        raise ValueError(
            f"Invalid value for f: {f}. Expected one of ['0', 'eom1.f', 'eom2.f', 'eom3.f']."
        )

    mir = port.split(".")[0]
    tmp_kat = kat.deepcopy()

    tmp_kat.parse(f"""
    ad {mir}_00 {port} {f} 0 0
    ad {mir}_10 {port} {f} 1 0
    ad {mir}_01 {port} {f} 0 1
    """)

    tmp_kat.parse("noxaxis(pre_step=none, post_step=none, name='noxaxis')")
    out = tmp_kat.run()

    dx = (
        abs(out[f"{mir}_10"])
        / abs(out[f"{mir}_00"])
        * bp[f"{port}"][0].w
        * 1e6
        * np.cos(np.angle(out[f"{mir}_10"]) - np.angle(out[f"{mir}_00"]))
    )
    dy = (
        abs(out[f"{mir}_01"])
        / abs(out[f"{mir}_00"])
        * bp[f"{port}"][1].w
        * 1e6
        * np.cos(np.angle(out[f"{mir}_01"]) - np.angle(out[f"{mir}_00"]))
    )

    return dx, dy


def make_SoS_QDs(Bx, kat, freq):
    # This function generates QDs, attached to the Bx port, at a certain 'freq' frequency
    # that should be as a beat note between sidebands. It adds the commands for finesse
    # to track the beat note and  also the _DC QDs used to tune the demodulation phase.
    #
    # The kat file should already contain the Bx_nf/ff_ports attached to appropriate telescopes.

    kat.add_frequency(freq)
    kat.add_frequency(-freq)

    kat.parse(f"""    
    pd1 {Bx}_nf_{freq * 1e-6:.0f}MHz_I {Bx}_nf_port.p1.i f={freq:.0f} phase=90  pdtype=xsplit
    pd1 {Bx}_nf_{freq * 1e-6:.0f}MHz_Q {Bx}_nf_port.p1.i f={freq:.0f} phase=0   pdtype=xsplit
    pd1 {Bx}_nf_{freq * 1e-6:.0f}MHz_I_DC {Bx}_nf_port.p1.i f={freq:.0f} phase=90 
    pd1 {Bx}_nf_{freq * 1e-6:.0f}MHz_Q_DC {Bx}_nf_port.p1.i f={freq:.0f} phase=0  
    
    pd1 {Bx}_ff_{freq * 1e-6:.0f}MHz_I {Bx}_ff_port.p1.i f={freq:.0f} phase=90  pdtype=xsplit
    pd1 {Bx}_ff_{freq * 1e-6:.0f}MHz_Q {Bx}_ff_port.p1.i f={freq:.0f} phase=0   pdtype=xsplit
    pd1 {Bx}_ff_{freq * 1e-6:.0f}MHz_I_DC {Bx}_ff_port.p1.i f={freq:.0f} phase=90 
    pd1 {Bx}_ff_{freq * 1e-6:.0f}MHz_Q_DC {Bx}_ff_port.p1.i f={freq:.0f} phase=0       
    """)


def make_2f_QDs(Bx, kat, freq):
    # This function generates QDs at twice the 'freq' frequency, attached to the Bx port.
    #
    # The kat file should already contain the Bx_nf/ff_ports attached to appropriate telescopes.

    kat.parse(f"""     
    pd1 {Bx}_nf_{2 * freq * 1e-6:.0f}MHz_I {Bx}_nf_port.p1.i f={2 * freq:.0f} phase=90  pdtype=xsplit
    pd1 {Bx}_nf_{2 * freq * 1e-6:.0f}MHz_Q {Bx}_nf_port.p1.i f={2 * freq:.0f} phase=0   pdtype=xsplit
    pd1 {Bx}_ff_{2 * freq * 1e-6:.0f}MHz_I {Bx}_ff_port.p1.i f={2 * freq:.0f} phase=90  pdtype=xsplit
    pd1 {Bx}_ff_{2 * freq * 1e-6:.0f}MHz_Q {Bx}_ff_port.p1.i f={2 * freq:.0f} phase=0   pdtype=xsplit      
    """)


def get_gouy(solution):
    # This function extract the accumulated gouy from the solution of the finesse tracing tool.
    # The beam should be traced from on of the ITMs towards the recycling mirror.

    segments = list(solution.all_segments().items())
    last_key, last_value = segments[-1]
    return last_value[1]["gouy"][-1]

In [ ]:
# create a new Finesse model object,
# this serves as the foundation of the simulation, so we'll name it `base`
base = finesse.Model()

# setup of the interferometer, each line is one component
# ITF version: IMCv3, SB1: 6.37 MHz, SB2: 27.61 MHz, SB3: 78.57 MHz, PRC: [35m, 25°], SRC: [38m, 19°]

PRCL0 = 35.2935
SRCL0 = 38.0083

base.parse_file("AdV_O5_baseline.kat")

base.phase_config(zero_k00=False)  # already the default
base.modes(maxtem=2)

In [ ]:
ns = base.ns

BSHR_to_NIHR = base.sBSsub1.L * ns + base.lBS_NI.L + base.sNIsub.L * ns
BSHR_to_WIHR = base.lBS_WI.L + base.sWIsub.L * ns

print("#### SBs check: main kat ####")
print(f"SB1: {base.eom1.f} Hz")
print(f"SB2: {base.eom2.f} Hz")
print(f"SB4: {base.eom4.f} Hz\n")
print(f"SB3: {base.eom3.f} Hz")
print("\n")
print("++++ PRC v8p1 (35.2935m, 26° Gouy) 2025/03/13 ++++")
print(f"PRM1 Rc: {base.PRM1.Rc} m")
print(f"PRM2 Rc: {base.PRM2.Rc} m")
print(f"PRM3 Rc: {base.PRM3.Rc} m\n")
print(f"PRM1 -> PRM2: {base.L1_PRC.L}")
print(f"PRM2 -> PRM3: {base.L2_PRC.L}")
print(f"PRM3 -> BS_HR: {base.L3_PRC.L}")
print(
    f"PRCL: {base.L1_PRC.L + base.L2_PRC.L + base.L3_PRC.L + (BSHR_to_NIHR + BSHR_to_WIHR) / 2} m (should be {PRCL0} m)"
)
print("\n")
print("++++ SRC v9p1 (38.0083m, 18° Gouy) 2025/03/13 ++++")
print(f"SRM1 Rc: {base.SRM1.Rc} m")
print(f"SRM2 Rc: {base.SRM2.Rc} m")
print(f"SRM3 Rc: {base.SRM3.Rc} m\n")
print(f"SRM3 -> BS_HR: {base.L3_SRC.L + ns * base.sBSsub2.L}")
print(f"SRM3 -> SRM2: {base.L2_SRC.L} m")
print(f"SRM2 -> SRM1: {base.L1_SRC.L} m")
print(
    f"SRCL: {base.L1_SRC.L + base.L2_SRC.L + base.L3_SRC.L + ns * base.sBSsub2.L + (BSHR_to_NIHR + BSHR_to_WIHR) / 2} m (should be {SRCL0} m)"
)
print("\n")
print(f"BS_HR -> NI_HR: {BSHR_to_NIHR}")
print(f"BS_HR -> WI_HR: {BSHR_to_WIHR}")
print("\n")

## Trace the beam and check the recycling cavities

In [ ]:
model = base.deepcopy()
beam_param = model.beam_trace()

In [ ]:
prop = finesse.tracing.tools.propagate_beam(
    from_node=model.WITM.p1.o,
    to_node=model.WETM.p1.o,
    direction="x",
)
prop.plot()
print(prop)

### Find Gouy phase at the arm's waist to define near field

In [ ]:
def cav_prop_trace(sol, prop):
    """
    This function takes the solution 'sol' of the beam tracing
    tool and the desired property 'prop' chosen among:

    - 'beamsize'
    - 'gouy'
    - 'curvature'

    and gives that property's evolution along the cavity axis.
    It's the same vector plotted by the propagate_beam() tool.
    """

    return list(sol.all_segments().items())[-1][1][1][prop]

In [ ]:
waist_idx = np.argmin(cav_prop_trace(prop, "beamsize"))
arm_gouy_trace = cav_prop_trace(prop, "gouy")

waist_gouy = arm_gouy_trace[waist_idx]
print(f"Gouy phase at the waist in the West arm: {waist_gouy:.4f} deg")

## Check working point

In [ ]:
model_dofs = base.deepcopy()
model_dofs.parse("""
dof MICH NITM.dofs.z -1 NIAR.dofs.z -1 NETM.dofs.z -1 WITM.dofs.z +1 WIAR.dofs.z +1 WETM.dofs.z +1
dof DARM NETM.dofs.z -1 WETM.dofs.z +1
dof CARM NETM.dofs.z +1 WETM.dofs.z +1
dof PRCL PRM1.dofs.z +1
dof SRCL SRM1.dofs.z -1

# Individual mirror motions; they are needed as a "dof" object is what we need to scan later
dof NE_z NETM.dofs.z +1
dof NI_z NITM.dofs.z +1
dof WE_z WETM.dofs.z +1
dof WI_z WITM.dofs.z +1
""")

model_dofs.parse_file("readout_rf.kat")
model_dofs.parse_file("readout_dc.kat")
model_dofs.parse_file("ad_plane_waves.kat")
model_dofs.parse_file("ad_HOMs.kat")
model_dofs.parse_file("pdh_for_TFs.kat")

## Pretuning

In [ ]:
## Run the model and make this changes in series:
model_tuned = model_dofs.deepcopy()

out1 = model_tuned.run(
    fac.TemporaryParameters(
        fac.Series(
            # Switch off the modulators and misalign SR and PR mirrors:
            fac.Change(
                {
                    "eom1.midx": 0,
                    "eom2.midx": 0,
                    "eom4.midx": 0,
                    "eom3.midx": 0,
                    "SRM1.misaligned": True,
                    "PRM1.misaligned": True,
                }
            ),
            # Maximize arm powers:
            fac.Maximize("B7_DC", "NE_z.DC", bounds=[-30, 30], tol=1e-14),
            fac.Maximize("B8_DC", "WE_z.DC", bounds=[-30, 30], tol=1e-14),
            # Minimize dark fringe power:
            fac.Minimize("B1p_DC", "MICH.DC", bounds=[-180, 180], tol=1e-14),
            # Align back PRM:
            fac.Change({"PRM1.misaligned": False}),
            # Maximize PRC power:
            fac.Maximize("B4_CAR_00", "PRCL.DC", bounds=[-180, 180], tol=1e-14),
            # Align back SRM:
            fac.Change({"SRM1.misaligned": False}),
            # Maximize SRC power, then offset by 90 deg:
            fac.Change({"SRCL.DC": 0}),
            fac.Maximize("B1p_CAR_00", "SRCL.DC", bounds=[-180, 180], tol=1e-14),
            fac.Change({"SRCL.DC": -90}, relative=True),
        ),
        # The 'exclude' parameter is needed as TemporaryParameters brings everything back to the starting point at the end of the .run
        exclude=(
            "PRM1.phi",
            "NITM.phi",
            "NETM.phi",
            "WITM.phi",
            "WETM.phi",
            "SRM1.phi",
            "NE_z.DC",
            "WE_z.DC",
            "MICH.DC",
            "PRCL.DC",
            "SRCL.DC",
        ),
    )
)

In [ ]:
print("PRM1 phi: ", model_tuned.PRM1.phi.eval())
print("SRM1 phi: ", model_tuned.SRM1.phi.eval())
print("WE phi: ", model_tuned.WETM.phi.eval())
print("WI phi: ", model_tuned.WITM.phi.eval())
print("NE phi: ", model_tuned.NETM.phi.eval())
print("NI phi: ", model_tuned.NITM.phi.eval())
print("MICH phi: ", model_tuned.MICH.DC)
print("PRCL phi: ", model_tuned.PRCL.DC)
print("SRCL phi: ", model_tuned.SRCL.DC)

## Check SB fields

In [ ]:
out_NARM = model_tuned.run("Xaxis(NE_z.DC, lin, -4, 4, 500, relative=True)")
out_WARM = model_tuned.run("Xaxis(WE_z.DC, lin, -4, 4, 500, relative=True)")
out_PRCL = model_tuned.run("Xaxis(PRCL.DC, lin, -120, 120, 500, relative=True)")
out_MICH = model_tuned.run("Xaxis(MICH.DC, lin, -15, 15, 500, relative=True)")
out_SRCL = model_tuned.run("Xaxis(SRCL.DC, lin, -120, 120, 500, relative=True)")

In [ ]:
model_tuned.modes_setting

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(10, 6))
ax[0, 0].semilogy(out_NARM.x1, out_NARM["B7_DC"], label="B7")
ax[0, 0].semilogy(out_WARM.x1, out_WARM["B8_DC"], label="B8")
ax[0, 0].margins(x=0)
ax[0, 0].legend(loc=3)
ax[0, 0].grid()
ax[0, 0].set_xlabel("Arms Sweep [deg]", fontsize=16)
ax[0, 0].set_ylabel(r"PDs [W]", fontsize=16)

ax[0, 1].semilogy(out_PRCL.x1, abs(out_PRCL["B4_USB1_00"]) ** 2, label="B4_USB1_00")
ax[0, 1].semilogy(out_PRCL.x1, abs(out_PRCL["B4_LSB1_00"]) ** 2, label="B4_LSB1_00")
ax[0, 1].semilogy(out_PRCL.x1, abs(out_PRCL["B4_USB2_00"]) ** 2, label="B4_USB2_00")
ax[0, 1].semilogy(out_PRCL.x1, abs(out_PRCL["B4_LSB2_00"]) ** 2, label="B4_LSB2_00")
ax[0, 1].semilogy(out_PRCL.x1, abs(out_PRCL["B4_DC"]), label="B4_DC")
ax[0, 1].margins(x=0)
ax[0, 1].legend(loc=3)
ax[0, 1].grid()
ax[0, 1].set_xlabel("PRCL Sweep [deg]", fontsize=16)
ax[0, 1].set_ylabel(r"Intra-PRC |ADs|$^2$ [W]", fontsize=16)

ax[1, 0].semilogy(out_MICH.x1, abs(out_MICH["B1p_USB1_00"]) ** 2, label="B1p_USB1_00")
ax[1, 0].semilogy(out_MICH.x1, abs(out_MICH["B1p_LSB1_00"]) ** 2, label="B1p_LSB1_00")
ax[1, 0].semilogy(out_MICH.x1, abs(out_MICH["B1p_USB2_00"]) ** 2, label="B1p_USB2_00")
ax[1, 0].semilogy(out_MICH.x1, abs(out_MICH["B1p_LSB2_00"]) ** 2, label="B1p_LSB2_00")
ax[1, 0].semilogy(out_MICH.x1, abs(out_MICH["B1p_CAR_00"]) ** 2, label="B1p_CAR_00")
ax[1, 0].margins(x=0)
ax[1, 0].legend(loc=3)
ax[1, 0].grid()
ax[1, 0].set_xlabel("MICH Sweep [deg]", fontsize=16)
ax[1, 0].set_ylabel(r"ASY |ADs|$^2$ [W]", fontsize=16)

ax[1, 1].semilogy(out_SRCL.x1, abs(out_SRCL["B1p_USB1_00"]) ** 2, label="B1p_USB1_00")
ax[1, 1].semilogy(out_SRCL.x1, abs(out_SRCL["B1p_LSB1_00"]) ** 2, label="B1p_LSB1_00")
ax[1, 1].semilogy(out_SRCL.x1, abs(out_SRCL["B1p_USB2_00"]) ** 2, label="B1p_USB2_00")
ax[1, 1].semilogy(out_SRCL.x1, abs(out_SRCL["B1p_LSB2_00"]) ** 2, label="B1p_LSB2_00")
ax[1, 1].semilogy(out_SRCL.x1, abs(out_SRCL["B1p_USB3_00"]) ** 2, label="B1p_USB3_00")
ax[1, 1].semilogy(out_SRCL.x1, abs(out_SRCL["B1p_LSB3_00"]) ** 2, label="B1p_LSB3_00")
ax[1, 1].semilogy(out_SRCL.x1, abs(out_SRCL["B1p_USB3_10"]) ** 2, label="B1p_USB3_10")
ax[1, 1].semilogy(out_SRCL.x1, abs(out_SRCL["B1p_LSB3_10"]) ** 2, label="B1p_LSB3_10")
ax[1, 1].semilogy(out_SRCL.x1, abs(out_SRCL["B1p_CAR_00"]) ** 2, label="B1p_CAR_00")
ax[1, 1].margins(x=0)
ax[1, 1].legend(loc=3)
ax[1, 1].grid()
ax[1, 1].set_xlabel("SRCL Sweep [deg]", fontsize=16)
ax[1, 1].set_ylabel(r"ASY |ADs|$^2$ [W]", fontsize=16)

plt.show()
fig.set_tight_layout(True)

## Matching

In [ ]:
match_kat = model_tuned.deepcopy()

In [ ]:
out_PRCL = match_kat.run("Xaxis(PRCL.DC, lin, -180, 180, 500, relative=True)")
out_MICH = match_kat.run("Xaxis(MICH.DC, lin, -180, 180, 500, relative=True)")
out_SRCL = match_kat.run("Xaxis(SRCL.DC, lin, -180, 180, 500, relative=True)")

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10, 8))
ax[0].plot(out_SRCL.x1, abs(out_PRCL["B4_DC"]), label="B4_DC")
ax[0].semilogy(out_PRCL.x1, abs(out_PRCL["B4_CAR_00"]) ** 2, label="B4_CAR_00")
ax[0].semilogy(out_PRCL.x1, abs(out_PRCL["B4_CAR_20"]) ** 2, label="B4_CAR_20")
ax[0].semilogy(out_PRCL.x1, abs(out_PRCL["B4_CAR_02"]) ** 2, label="B4_CAR_02")
ax[0].semilogy(out_PRCL.x1, abs(out_PRCL["B4_USB1_00"]) ** 2, label="B4_USB1_00")
ax[0].semilogy(out_PRCL.x1, abs(out_PRCL["B4_LSB1_00"]) ** 2, label="B4_LSB1_00")
ax[0].semilogy(out_PRCL.x1, abs(out_PRCL["B4_USB1_20"]) ** 2, label="B4_USB1_20")
ax[0].semilogy(out_PRCL.x1, abs(out_PRCL["B4_LSB1_02"]) ** 2, label="B4_LSB1_02")
ax[0].semilogy(out_PRCL.x1, abs(out_PRCL["B4_USB2_00"]) ** 2, label="B4_USB2_00")
ax[0].semilogy(out_PRCL.x1, abs(out_PRCL["B4_LSB2_00"]) ** 2, label="B4_LSB2_00")
ax[0].semilogy(out_PRCL.x1, abs(out_PRCL["B4_USB2_20"]) ** 2, label="B4_USB2_20")
ax[0].semilogy(out_PRCL.x1, abs(out_PRCL["B4_LSB2_02"]) ** 2, label="B4_LSB2_02")
ax[0].semilogy(out_PRCL.x1, abs(out_PRCL["B4_USB4_00"]) ** 2, label="B4_USB4_00")
ax[0].semilogy(out_PRCL.x1, abs(out_PRCL["B4_LSB4_00"]) ** 2, label="B4_LSB4_00")
ax[0].margins(x=0)
ax[0].legend(loc=3)
ax[0].grid()
ax[0].set_xlabel("PRCL Sweep [deg]", fontsize=16)
ax[0].set_ylabel(r"PDs [W]", fontsize=16)

ax[1].plot(out_SRCL.x1, abs(out_SRCL["B1p_DC"]), label="B1p_DC")
ax[1].semilogy(out_SRCL.x1, abs(out_SRCL["B1p_CAR_00"]) ** 2, label="B1p_CAR_00")
ax[1].semilogy(out_SRCL.x1, abs(out_SRCL["B1p_CAR_20"]) ** 2, label="B1p_CAR_20")
ax[1].semilogy(out_SRCL.x1, abs(out_SRCL["B1p_CAR_02"]) ** 2, label="B1p_CAR_02")
ax[1].semilogy(out_SRCL.x1, abs(out_SRCL["B1p_USB1_00"]) ** 2, label="B1p_USB1_00")
ax[1].semilogy(out_SRCL.x1, abs(out_SRCL["B1p_LSB1_00"]) ** 2, label="B1p_LSB1_00")
ax[1].semilogy(out_SRCL.x1, abs(out_SRCL["B1p_USB1_20"]) ** 2, label="B1p_USB1_20")
ax[1].semilogy(out_SRCL.x1, abs(out_SRCL["B1p_LSB1_02"]) ** 2, label="B1p_LSB1_02")
ax[1].semilogy(out_SRCL.x1, abs(out_SRCL["B1p_USB2_00"]) ** 2, label="B1p_USB2_00")
ax[1].semilogy(out_SRCL.x1, abs(out_SRCL["B1p_LSB2_00"]) ** 2, label="B1p_LSB2_00")
ax[1].semilogy(out_SRCL.x1, abs(out_SRCL["B1p_USB2_20"]) ** 2, label="B1p_USB2_20")
ax[1].semilogy(out_SRCL.x1, abs(out_SRCL["B1p_LSB2_02"]) ** 2, label="B1p_LSB2_02")
ax[1].margins(x=0)
ax[1].legend(loc=3)
ax[1].grid()
ax[1].set_xlabel("SRCL Sweep [deg]", fontsize=16)
ax[1].set_ylabel(r"ADs [W$^{1/2}$]", fontsize=16)
plt.show()
fig.set_tight_layout(True)

## Run a simple simulations to find the DC power impinging on each PD for normalization purposes

In [ ]:
model_dof_PDs = model_tuned.deepcopy()
model_dof_PDs.parse("noxaxis()")
out_PDs = model_dof_PDs.run()

In [ ]:
print(f"B7_DC: {out_PDs['B7_DC'] * 1e3:.2f} mW")
print(f"B8_DC: {out_PDs['B8_DC'] * 1e3:.2f} mW")
print(f"B2_DC: {out_PDs['B2_DC'] * 1e3:.2f} mW")
print(f"B4_DC: {out_PDs['B4_DC'] * 1e3:.2f} mW")
print(f"B1p_DC: {out_PDs['B1p_DC'] * 1e3:.2f} mW")

## Now we are going to check the DARM TF

In [ ]:
model_dof_DARMcheck = model_tuned.deepcopy()

model_dof_DARMcheck.parse("""
fsig(1) 
sgen test1 DARM.AC
pd1 B1p_TF SRM1.p2.o f=fsig.f
xaxis(fsig.f , log, 1, 1000, 100)
""")

out = model_dof_DARMcheck.run()

fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.loglog(out.x1, np.abs(out["B1p_TF"]), label="B1p_DC")
ax.grid()
ax.legend()
ax.set_xlabel("f [Hz]", fontsize=16)
ax.set_ylabel("abs [W/m", fontsize=16)

fig.set_tight_layout(True)

### We start by shaking DARM

In [ ]:
model_dof_DARMsens = model_tuned.deepcopy()

model_dof_DARMsens.parse("""
fsig(1) 
sgen test1 DARM.AC
xaxis(fsig.f , log, 1e-3, 1e3, 10)
""")

In [ ]:
DARM = model_dof_DARMsens.run()

fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.loglog(DARM.x1, np.abs(DARM["B2_SB1_I_TF"]), label="B2_SB1_I")
ax.grid()
ax.legend()
ax.set_xlabel("f [Hz]", fontsize=16)
ax.set_ylabel("abs [W/m", fontsize=16)

fig.set_tight_layout(True)

In [ ]:
u_darm = [
    DARM["B2_SB1_Q_TF"][1].real,
    DARM["B2_SB2_Q_TF"][1].real,
    DARM["B4_SB1_Q_TF"][1].real,
    DARM["B4_SB2_Q_TF"][1].real,
    DARM["B6_SB1_Q_TF"][1].real,
    DARM["B6_SB2_Q_TF"][1].real,
    DARM["B1p_SB1_Q_TF"][1].real,
    DARM["B1p_SB2_Q_TF"][1].real,
]
v_darm = [
    DARM["B2_SB1_I_TF"][1].real,
    DARM["B2_SB2_I_TF"][1].real,
    DARM["B4_SB1_I_TF"][1].real,
    DARM["B4_SB2_I_TF"][1].real,
    DARM["B6_SB1_I_TF"][1].real,
    DARM["B6_SB2_I_TF"][1].real,
    DARM["B1p_SB1_I_TF"][1].real,
    DARM["B1p_SB2_I_TF"][1].real,
]

In [ ]:
angles_darm, radii_darm_mag = cart2pol(u_darm, v_darm)
radii_darm_mag_norm = [
    radii_darm_mag[0] / out_PDs["B2_DC"],
    radii_darm_mag[1] / out_PDs["B2_DC"],
    radii_darm_mag[2] / out_PDs["B4_DC"],
    radii_darm_mag[3] / out_PDs["B4_DC"],
    radii_darm_mag[4] / out_PDs["B6_DC"],
    radii_darm_mag[5] / out_PDs["B6_DC"],
    radii_darm_mag[6] / out_PDs["B1p_DC"],
    radii_darm_mag[7] / out_PDs["B1p_DC"],
]
radii_darm = 20 * np.log10(radii_darm_mag)

# sns.set_palette(palette='Paired', n_colors=8, color_codes=True)
# clr=['b','g','r','c','m','y','k','w']
clr = [
    "tab:orange",
    "tab:pink",
    "tab:olive",
    "tab:purple",
    "tab:green",
    "tab:cyan",
    "tab:blue",
    "tab:red",
]

signals = [
    "B2 SB1",
    "B2 SB2",
    "B4 SB1",
    "B4 SB2",
    "B6 SB1",
    "B6 SB2",
    "B1 SB1",
    "B1 SB2",
]

fig, ax = plt.subplots(1, 1, figsize=(7, 5), subplot_kw={"projection": "polar"})
fig.patch.set_facecolor("white")
kw = dict(arrowstyle="->")

for x in range(0, len(angles_darm)):
    plt.polar(angles_darm[x], radii_darm[x], label=signals[x], color=clr[x])
    kw.update(arrowstyle="->", color=clr[x])
    [ax.annotate("", xy=(angles_darm[x], radii_darm[x]), xytext=(0, 0), arrowprops=kw)]

ax.set_ylim(0, np.max(radii_darm))
plt.title("Compass plot for DARM")
plt.legend(loc="lower left")

In [ ]:
# assign data
mydata = [
    [
        "OG [dB]",
        radii_darm[0],
        radii_darm[1],
        radii_darm[2],
        radii_darm[3],
        radii_darm[4],
        radii_darm[5],
        radii_darm[6],
        radii_darm[7],
    ],
    [
        "OG [mag]",
        radii_darm_mag[0],
        radii_darm_mag[1],
        radii_darm_mag[2],
        radii_darm_mag[3],
        radii_darm_mag[4],
        radii_darm_mag[5],
        radii_darm_mag[6],
        radii_darm_mag[7],
    ],
    [
        "OG norm [mag]",
        radii_darm_mag_norm[0],
        radii_darm_mag_norm[1],
        radii_darm_mag_norm[2],
        radii_darm_mag_norm[3],
        radii_darm_mag_norm[4],
        radii_darm_mag_norm[5],
        radii_darm_mag_norm[6],
        radii_darm_mag_norm[7],
    ],
    [
        "Phi",
        angles_darm[0] * 180 / np.pi,
        angles_darm[1] * 180 / np.pi,
        angles_darm[2] * 180 / np.pi,
        angles_darm[3] * 180 / np.pi,
        angles_darm[4] * 180 / np.pi,
        angles_darm[5] * 180 / np.pi,
        angles_darm[6] * 180 / np.pi,
        angles_darm[7] * 180 / np.pi,
    ],
]
# create header
head = ["B2 SB1", "B2 SB2", "B4 SB1", "B4 SB2", "B6 SB1", "B6 SB2", "B1 SB1", "B1 SB2"]

# display table
print(tabulate(mydata, headers=head, tablefmt="grid"))

### Now we scan CARM DOF

In [ ]:
model_dof_CARMsens = model_tuned.deepcopy()

model_dof_CARMsens.parse("""
fsig(1) 
sgen test1 CARM.AC
xaxis(fsig.f , log, 1e-3, 1e3, 10)
#xaxis(fsig.f , log, 1e-2, 1000, 100)
""")

In [ ]:
CARM = model_dof_CARMsens.run()

fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.loglog(CARM.x1, np.abs(CARM["B4_SB2_Q_TF"]), label="B4_SB2_Q_DC")
ax.grid()
ax.legend()
ax.set_xlabel("f [Hz]", fontsize=16)
ax.set_ylabel("abs [W/m", fontsize=16)

fig.set_tight_layout(True)

In [ ]:
u_carm = [
    CARM["B2_SB1_Q_TF"][1].real,
    CARM["B2_SB2_Q_TF"][1].real,
    CARM["B4_SB1_Q_TF"][1].real,
    CARM["B4_SB2_Q_TF"][1].real,
    CARM["B6_SB1_Q_TF"][1].real,
    CARM["B6_SB2_Q_TF"][1].real,
    CARM["B1p_SB1_Q_TF"][1].real,
    CARM["B1p_SB2_Q_TF"][1].real,
]
v_carm = [
    CARM["B2_SB1_I_TF"][1].real,
    CARM["B2_SB2_I_TF"][1].real,
    CARM["B4_SB1_I_TF"][1].real,
    CARM["B4_SB2_I_TF"][1].real,
    CARM["B6_SB1_I_TF"][1].real,
    CARM["B6_SB2_I_TF"][1].real,
    CARM["B1p_SB1_I_TF"][1].real,
    CARM["B1p_SB2_I_TF"][1].real,
]

In [ ]:
angles_carm, radii_carm_mag = cart2pol(u_carm, v_carm)
radii_carm_mag_norm = [
    radii_carm_mag[0] / out_PDs["B2_DC"],
    radii_carm_mag[1] / out_PDs["B2_DC"],
    radii_carm_mag[2] / out_PDs["B4_DC"],
    radii_carm_mag[3] / out_PDs["B4_DC"],
    radii_carm_mag[4] / out_PDs["B6_DC"],
    radii_carm_mag[5] / out_PDs["B6_DC"],
    radii_carm_mag[6] / out_PDs["B1p_DC"],
    radii_carm_mag[7] / out_PDs["B1p_DC"],
]
radii_carm = 20 * np.log10(radii_carm_mag)

# sns.set_palette(palette='Paired', n_colors=8, color_codes=True)
# clr=['b','g','r','c','m','y','k','w']
clr = [
    "tab:orange",
    "tab:pink",
    "tab:olive",
    "tab:purple",
    "tab:green",
    "tab:cyan",
    "tab:blue",
    "tab:red",
]

signals = [
    "B2 SB1",
    "B2 SB2",
    "B4 SB1",
    "B4 SB2",
    "B6 SB1",
    "B6 SB2",
    "B1 SB1",
    "B1 SB2",
]

fig, ax = plt.subplots(1, 1, figsize=(7, 5), subplot_kw={"projection": "polar"})
fig.patch.set_facecolor("white")
kw = dict(arrowstyle="->")

for x in range(0, len(angles_carm)):
    plt.polar(angles_carm[x], radii_carm[x], label=signals[x], color=clr[x])
    kw.update(arrowstyle="->", color=clr[x])
    [ax.annotate("", xy=(angles_carm[x], radii_carm[x]), xytext=(0, 0), arrowprops=kw)]

ax.set_ylim(0, np.max(radii_carm))
plt.title("Compass plot for CARM")
plt.legend(loc="lower left")

In [ ]:
# assign data
mydata = [
    [
        "OG [dB]",
        radii_carm[0],
        radii_carm[1],
        radii_carm[2],
        radii_carm[3],
        radii_carm[4],
        radii_carm[5],
        radii_carm[6],
        radii_carm[7],
    ],
    [
        "OG [mag]",
        radii_carm_mag[0],
        radii_carm_mag[1],
        radii_carm_mag[2],
        radii_carm_mag[3],
        radii_carm_mag[4],
        radii_carm_mag[5],
        radii_carm_mag[6],
        radii_carm_mag[7],
    ],
    [
        "OG norm [mag]",
        radii_carm_mag_norm[0],
        radii_carm_mag_norm[1],
        radii_carm_mag_norm[2],
        radii_carm_mag_norm[3],
        radii_carm_mag_norm[4],
        radii_carm_mag_norm[5],
        radii_carm_mag_norm[6],
        radii_carm_mag_norm[7],
    ],
    [
        "Phi",
        angles_carm[0] * 180 / np.pi,
        angles_carm[1] * 180 / np.pi,
        angles_carm[2] * 180 / np.pi,
        angles_carm[3] * 180 / np.pi,
        angles_carm[4] * 180 / np.pi,
        angles_carm[5] * 180 / np.pi,
        angles_carm[6] * 180 / np.pi,
        angles_carm[7] * 180 / np.pi,
    ],
]
# create header
head = ["B2 SB1", "B2 SB2", "B4 SB1", "B4 SB2", "B6 SB1", "B6 SB2", "B1 SB1", "B1 SB2"]

# display table
print(tabulate(mydata, headers=head, tablefmt="grid"))

### Now we shake PRCL DOF

In [ ]:
model_dof_PRCLsens = model_tuned.deepcopy()

model_dof_PRCLsens.parse("""
fsig(1) 
sgen test1 PRCL.AC
xaxis(fsig.f , log, 1e-1, 1e4, 1000)

""")

In [ ]:
PRCL = model_dof_PRCLsens.run()

fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.loglog(PRCL.x1, np.abs(PRCL["B2_SB1_I_TF"]), label="B2_SB1_I")
ax.grid()
ax.legend()
ax.set_xlabel("f [Hz]", fontsize=16)
ax.set_ylabel("abs [W/m", fontsize=16)

fig.set_tight_layout(True)

In [ ]:
u_prcl = [
    PRCL["B2_SB1_Q_TF"][1].real,
    PRCL["B2_SB2_Q_TF"][1].real,
    PRCL["B4_SB1_Q_TF"][1].real,
    PRCL["B4_SB2_Q_TF"][1].real,
    PRCL["B6_SB1_Q_TF"][1].real,
    PRCL["B6_SB2_Q_TF"][1].real,
    PRCL["B1p_SB1_Q_TF"][1].real,
    PRCL["B1p_SB2_Q_TF"][1].real,
]
v_prcl = [
    PRCL["B2_SB1_I_TF"][1].real,
    PRCL["B2_SB2_I_TF"][1].real,
    PRCL["B4_SB1_I_TF"][1].real,
    PRCL["B4_SB2_I_TF"][1].real,
    PRCL["B6_SB1_I_TF"][1].real,
    PRCL["B6_SB2_I_TF"][1].real,
    PRCL["B1p_SB1_I_TF"][1].real,
    PRCL["B1p_SB2_I_TF"][1].real,
]

In [ ]:
angles_prcl, radii_prcl_mag = cart2pol(u_prcl, v_prcl)
radii_prcl_mag_norm = [
    radii_prcl_mag[0] / out_PDs["B2_DC"],
    radii_prcl_mag[1] / out_PDs["B2_DC"],
    radii_prcl_mag[2] / out_PDs["B4_DC"],
    radii_prcl_mag[3] / out_PDs["B4_DC"],
    radii_prcl_mag[4] / out_PDs["B6_DC"],
    radii_prcl_mag[5] / out_PDs["B6_DC"],
    radii_prcl_mag[6] / out_PDs["B1p_DC"],
    radii_prcl_mag[7] / out_PDs["B1p_DC"],
]
radii_prcl = 20 * np.log10(radii_prcl_mag)

# sns.set_palette(palette='Paired', n_colors=8, color_codes=True)
# clr=['b','g','r','c','m','y','k','w']
clr = [
    "tab:orange",
    "tab:pink",
    "tab:olive",
    "tab:purple",
    "tab:green",
    "tab:cyan",
    "tab:blue",
    "tab:red",
]

signals = [
    "B2 SB1",
    "B2 SB2",
    "B4 SB1",
    "B4 SB2",
    "B6 SB1",
    "B6 SB2",
    "B1 SB1",
    "B1 SB2",
]

fig, ax = plt.subplots(1, 1, figsize=(7, 5), subplot_kw={"projection": "polar"})
fig.patch.set_facecolor("white")
kw = dict(arrowstyle="->")

for x in range(0, len(angles_prcl)):
    plt.polar(angles_prcl[x], radii_prcl[x], label=signals[x], color=clr[x])
    kw.update(arrowstyle="->", color=clr[x])
    [ax.annotate("", xy=(angles_prcl[x], radii_prcl[x]), xytext=(0, 0), arrowprops=kw)]

ax.set_ylim(0, np.max(radii_prcl))
plt.title("Compass plot for PRCL")
plt.legend(loc="lower left")

In [ ]:
# assign data
mydata = [
    [
        "OG [dB]",
        radii_prcl[0],
        radii_prcl[1],
        radii_prcl[2],
        radii_prcl[3],
        radii_prcl[4],
        radii_prcl[5],
        radii_prcl[6],
        radii_prcl[7],
    ],
    [
        "OG [mag]",
        radii_prcl_mag[0],
        radii_prcl_mag[1],
        radii_prcl_mag[2],
        radii_prcl_mag[3],
        radii_prcl_mag[4],
        radii_prcl_mag[5],
        radii_prcl_mag[6],
        radii_prcl_mag[7],
    ],
    [
        "OG norm [mag]",
        radii_prcl_mag_norm[0],
        radii_prcl_mag_norm[1],
        radii_prcl_mag_norm[2],
        radii_prcl_mag_norm[3],
        radii_prcl_mag_norm[4],
        radii_prcl_mag_norm[5],
        radii_prcl_mag_norm[6],
        radii_prcl_mag_norm[7],
    ],
    [
        "Phi",
        angles_prcl[0] * 180 / np.pi,
        angles_prcl[1] * 180 / np.pi,
        angles_prcl[2] * 180 / np.pi,
        angles_prcl[3] * 180 / np.pi,
        angles_prcl[4] * 180 / np.pi,
        angles_prcl[5] * 180 / np.pi,
        angles_prcl[6] * 180 / np.pi,
        angles_prcl[7] * 180 / np.pi,
    ],
]
# create header
head = ["B2 SB1", "B2 SB2", "B4 SB1", "B4 SB2", "B6 SB1", "B6 SB2", "B1 SB1", "B1 SB2"]

# display table
print(tabulate(mydata, headers=head, tablefmt="grid"))

### Now we shake MICH

In [ ]:
model_dof_MICHsens = model_tuned.deepcopy()

model_dof_MICHsens.parse("""
fsig(1) 
sgen test1 MICH.AC
xaxis(fsig.f , log, 1e-1, 1e4, 1000)
""")

In [ ]:
MICH = model_dof_MICHsens.run()

fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.loglog(MICH.x1, np.abs(MICH["B4_SB2_I_TF"]), label="B4_SB2_I")
ax.grid()
ax.legend()
ax.set_xlabel("f [Hz]", fontsize=16)
ax.set_ylabel("abs [W/m", fontsize=16)

fig.set_tight_layout(True)

In [ ]:
u_mich = [
    MICH["B2_SB1_Q_TF"][1].real,
    MICH["B2_SB2_Q_TF"][1].real,
    MICH["B4_SB1_Q_TF"][1].real,
    MICH["B4_SB2_Q_TF"][1].real,
    MICH["B6_SB1_Q_TF"][1].real,
    MICH["B6_SB2_Q_TF"][1].real,
    MICH["B1p_SB1_Q_TF"][1].real,
    MICH["B1p_SB2_Q_TF"][1].real,
]
v_mich = [
    MICH["B2_SB1_I_TF"][1].real,
    MICH["B2_SB2_I_TF"][1].real,
    MICH["B4_SB1_I_TF"][1].real,
    MICH["B4_SB2_I_TF"][1].real,
    MICH["B6_SB1_I_TF"][1].real,
    MICH["B6_SB2_I_TF"][1].real,
    MICH["B1p_SB1_I_TF"][1].real,
    MICH["B1p_SB2_I_TF"][1].real,
]

In [ ]:
angles_mich, radii_mich_mag = cart2pol(u_mich, v_mich)
radii_mich_mag_norm = [
    radii_mich_mag[0] / out_PDs["B2_DC"],
    radii_mich_mag[1] / out_PDs["B2_DC"],
    radii_mich_mag[2] / out_PDs["B4_DC"],
    radii_mich_mag[3] / out_PDs["B4_DC"],
    radii_mich_mag[4] / out_PDs["B6_DC"],
    radii_mich_mag[5] / out_PDs["B6_DC"],
    radii_mich_mag[6] / out_PDs["B1p_DC"],
    radii_mich_mag[7] / out_PDs["B1p_DC"],
]
radii_mich = 20 * np.log10(radii_mich_mag)

# sns.set_palette(palette='Paired', n_colors=8, color_codes=True)
# clr=['b','g','r','c','m','y','k','w']
clr = [
    "tab:orange",
    "tab:pink",
    "tab:olive",
    "tab:purple",
    "tab:green",
    "tab:cyan",
    "tab:blue",
    "tab:red",
]

signals = [
    "B2 SB1",
    "B2 SB2",
    "B4 SB1",
    "B4 SB2",
    "B6 SB1",
    "B6 SB2",
    "B1 SB1",
    "B1 SB2",
]

fig, ax = plt.subplots(1, 1, figsize=(7, 5), subplot_kw={"projection": "polar"})
fig.patch.set_facecolor("white")
kw = dict(arrowstyle="->")

for x in range(0, len(angles_mich)):
    plt.polar(angles_mich[x], radii_mich[x], label=signals[x], color=clr[x])
    kw.update(arrowstyle="->", color=clr[x])
    [ax.annotate("", xy=(angles_mich[x], radii_mich[x]), xytext=(0, 0), arrowprops=kw)]

ax.set_ylim(0, np.max(radii_mich))
plt.title("Compass plot for MICH")
plt.legend(loc="lower left")

In [ ]:
# assign data
mydata = [
    [
        "OG [dB]",
        radii_mich[0],
        radii_mich[1],
        radii_mich[2],
        radii_mich[3],
        radii_mich[4],
        radii_mich[5],
        radii_mich[6],
        radii_mich[7],
    ],
    [
        "OG [mag]",
        radii_mich_mag[0],
        radii_mich_mag[1],
        radii_mich_mag[2],
        radii_mich_mag[3],
        radii_mich_mag[4],
        radii_mich_mag[5],
        radii_mich_mag[6],
        radii_mich_mag[7],
    ],
    [
        "OG norm [mag]",
        radii_mich_mag_norm[0],
        radii_mich_mag_norm[1],
        radii_mich_mag_norm[2],
        radii_mich_mag_norm[3],
        radii_mich_mag_norm[4],
        radii_mich_mag_norm[5],
        radii_mich_mag_norm[6],
        radii_mich_mag_norm[7],
    ],
    [
        "Phi",
        angles_mich[0] * 180 / np.pi,
        angles_mich[1] * 180 / np.pi,
        angles_mich[2] * 180 / np.pi,
        angles_mich[3] * 180 / np.pi,
        angles_mich[4] * 180 / np.pi,
        angles_mich[5] * 180 / np.pi,
        angles_mich[6] * 180 / np.pi,
        angles_mich[7] * 180 / np.pi,
    ],
]
# create header
head = ["B2 SB1", "B2 SB2", "B4 SB1", "B4 SB2", "B6 SB1", "B6 SB2", "B1 SB1", "B1 SB2"]
# display table
print(tabulate(mydata, headers=head, tablefmt="grid"))

### Now we shake SRCL

In [ ]:
model_dof_SRCLsens = model_tuned.deepcopy()

model_dof_SRCLsens.parse("""
fsig(1) 
sgen test1 SRCL.AC
xaxis(fsig.f , log, 1e-1, 1e4, 100)
""")

In [ ]:
SRCL = model_dof_SRCLsens.run()

fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.loglog(SRCL.x1, np.abs(SRCL["B4_SB2_Q_TF"]), label="B4_SB2_Q")
ax.grid()
ax.legend()
ax.set_xlabel("f [Hz]", fontsize=16)
ax.set_ylabel("abs [W/m", fontsize=16)

fig.set_tight_layout(True)

In [ ]:
u_srcl = [
    SRCL["B2_SB1_Q_TF"][1].real,
    SRCL["B2_SB2_Q_TF"][1].real,
    SRCL["B4_SB1_Q_TF"][1].real,
    SRCL["B4_SB2_Q_TF"][1].real,
    SRCL["B6_SB1_Q_TF"][1].real,
    SRCL["B6_SB2_Q_TF"][1].real,
    SRCL["B1p_SB1_Q_TF"][1].real,
    SRCL["B1p_SB2_Q_TF"][1].real,
]
v_srcl = [
    SRCL["B2_SB1_I_TF"][1].real,
    SRCL["B2_SB2_I_TF"][1].real,
    SRCL["B4_SB1_I_TF"][1].real,
    SRCL["B4_SB2_I_TF"][1].real,
    SRCL["B6_SB1_I_TF"][1].real,
    SRCL["B6_SB2_I_TF"][1].real,
    SRCL["B1p_SB1_I_TF"][1].real,
    SRCL["B1p_SB2_I_TF"][1].real,
]

In [ ]:
angles_srcl, radii_srcl_mag = cart2pol(u_srcl, v_srcl)
radii_srcl_mag_norm = [
    radii_srcl_mag[0] / out_PDs["B2_DC"],
    radii_srcl_mag[1] / out_PDs["B2_DC"],
    radii_srcl_mag[2] / out_PDs["B4_DC"],
    radii_srcl_mag[3] / out_PDs["B4_DC"],
    radii_srcl_mag[4] / out_PDs["B6_DC"],
    radii_srcl_mag[5] / out_PDs["B6_DC"],
    radii_srcl_mag[6] / out_PDs["B1p_DC"],
    radii_srcl_mag[7] / out_PDs["B1p_DC"],
]
radii_srcl = 20 * np.log10(radii_srcl_mag)

# sns.set_palette(palette='Paired', n_colors=8, color_codes=True)
# clr=['b','g','r','c','m','y','k','w']
clr = [
    "tab:orange",
    "tab:pink",
    "tab:olive",
    "tab:purple",
    "tab:green",
    "tab:cyan",
    "tab:blue",
    "tab:red",
]

signals = [
    "B2 SB1",
    "B2 SB2",
    "B4 SB1",
    "B4 SB2",
    "B6 SB1",
    "B6 SB2",
    "B1 SB1",
    "B1 SB2",
]

fig, ax = plt.subplots(1, 1, figsize=(7, 5), subplot_kw={"projection": "polar"})
fig.patch.set_facecolor("white")
kw = dict(arrowstyle="->")

for x in range(0, len(angles_srcl)):
    plt.polar(angles_srcl[x], radii_srcl[x], label=signals[x], color=clr[x])
    kw.update(arrowstyle="->", color=clr[x])
    [ax.annotate("", xy=(angles_srcl[x], radii_srcl[x]), xytext=(0, 0), arrowprops=kw)]

ax.set_ylim(0, np.max(radii_srcl))
plt.title("Compass plot for SRCL")
plt.legend(loc="lower left")

In [ ]:
# assign data
mydata = [
    [
        "OG [dB]",
        radii_srcl[0],
        radii_srcl[1],
        radii_srcl[2],
        radii_srcl[3],
        radii_srcl[4],
        radii_srcl[5],
        radii_srcl[6],
        radii_srcl[7],
    ],
    [
        "OG [mag]",
        radii_srcl_mag[0],
        radii_srcl_mag[1],
        radii_srcl_mag[2],
        radii_srcl_mag[3],
        radii_srcl_mag[4],
        radii_srcl_mag[5],
        radii_srcl_mag[6],
        radii_srcl_mag[7],
    ],
    [
        "OG norm [mag]",
        radii_srcl_mag_norm[0],
        radii_srcl_mag_norm[1],
        radii_srcl_mag_norm[2],
        radii_srcl_mag_norm[3],
        radii_srcl_mag_norm[4],
        radii_srcl_mag_norm[5],
        radii_srcl_mag_norm[6],
        radii_srcl_mag_norm[7],
    ],
    [
        "Phi",
        angles_srcl[0] * 180 / np.pi,
        angles_srcl[1] * 180 / np.pi,
        angles_srcl[2] * 180 / np.pi,
        angles_srcl[3] * 180 / np.pi,
        angles_srcl[4] * 180 / np.pi,
        angles_srcl[5] * 180 / np.pi,
        angles_srcl[6] * 180 / np.pi,
        angles_srcl[7] * 180 / np.pi,
    ],
]
# create header
head = ["B2 SB1", "B2 SB2", "B4 SB1", "B4 SB2", "B6 SB1", "B6 SB2", "B1 SB1", "B1 SB2"]

# display table
print(tabulate(mydata, headers=head, tablefmt="grid"))


# We start building the longitudinal sensing matrix

## Let us prepare the final block of sensors with the correct demodulation phases

In [ ]:
matrix_kat = model_tuned.deepcopy()

matrix_kat.B2_SB1.phase = 90 - angles_prcl[0] * 180 / np.pi
matrix_kat.B4_SB2.phase = 90 - angles_mich[3] * 180 / np.pi
matrix_kat.B1p_SB2.phase = 90 - angles_darm[7] * 180 / np.pi
matrix_kat.B4_SB1.phase = 90 - angles_carm[2] * 180 / np.pi

print(matrix_kat.B2_SB1.phase)
print(matrix_kat.B4_SB2.phase)
print(matrix_kat.B1p_SB2.phase)
print(matrix_kat.B4_SB1.phase)

## Check PDH Signals

In [ ]:
# Timo: Increased to 10000 points for a more accurate optical gain calculation.
# With only 100 points as before the numerical OG calculation was off by up to factor
# of 3.
kat_P = matrix_kat.deepcopy()
kat_P.parse("xaxis(PRCL.DC, lin, -10, 10, 10000, relative=True)")
out_P = kat_P.run()

kat_M = matrix_kat.deepcopy()
kat_M.parse("xaxis(MICH.DC, lin, -10, 10, 10000, relative=True)")
out_M = kat_M.run()

kat_S = matrix_kat.deepcopy()
kat_S.parse("xaxis(SRCL.DC, lin, -10, 10, 10000, relative=True)")
out_S = kat_S.run()

kat_C = matrix_kat.deepcopy()
kat_C.parse("xaxis(CARM.DC, lin, -0.25, 0.25, 10000, relative=True)")
out_C = kat_C.run()

kat_D = matrix_kat.deepcopy()
kat_D.parse("xaxis(DARM.DC, lin, -10, 10, 10000, relative=True)")
out_D = kat_D.run()

In [ ]:
fig, ax = plt.subplots(5, 1, figsize=(8, 15))

ax[0].plot(out_P.x1, np.real(out_P["B2_SB1_I"]), label="B2_SB1_I")
ax[0].plot(out_P.x1, np.real(out_P["B2_SB1_Q"]), label="B2_SB1_Q")
ax[0].plot(
    out_P.x1, np.real(out_P["B4_DC"]) / 1000, label="B4_DC * 0.001", linestyle="--"
)
ax[0].set_ylabel("PRCL", fontsize=20)
ax[0].grid()
ax[0].legend()

ax[1].plot(out_M.x1, np.real(out_M["B4_SB2_I"]), label="B4_SB2_I")
ax[1].plot(out_M.x1, np.real(out_M["B4_SB2_Q"]), label="B4_SB2_Q")
ax[1].plot(
    out_M.x1, np.real(out_M["B1p_DC"] / 100), label="B1p_DC*0.01", linestyle="--"
)
ax[1].set_ylabel("MICH", fontsize=20)
ax[1].grid()
ax[1].legend()

ax[2].plot(out_S.x1, np.real(out_S["B4_SB2_I"]), label="B4_SB2_I")
ax[2].plot(out_S.x1, np.real(out_S["B4_SB2_Q"]), label="B4_SB2_Q")
ax[2].plot(
    out_S.x1,
    np.abs(out_S["B1p_USB2_00"]) ** 2 / 100,
    label="B1p_USB2_00*0.01",
    linestyle="--",
)
ax[2].plot(
    out_S.x1,
    np.abs(out_S["B1p_LSB2_00"]) ** 2 / 100,
    label="B1p_LSB2_00*0.01",
    linestyle="--",
)
ax[2].set_ylabel("SRCL", fontsize=20)
ax[2].grid()
ax[2].legend()

ax[3].plot(out_C.x1, np.real(out_C["B4_SB1_I"]), label="B4_SB1_I")
ax[3].plot(out_C.x1, np.real(out_C["B4_SB1_Q"]), label="B4_SB1_Q")
ax[3].plot(out_C.x1, np.real(out_C["B7_DC"] / 100), label="B7_DC*0.01", linestyle="--")
ax[3].plot(out_C.x1, np.real(out_C["B8_DC"] / 100), label="B8_DC*0.01", linestyle="--")
ax[3].set_ylabel("CARM", fontsize=20)
ax[3].grid()
ax[3].legend()

ax[4].plot(out_D.x1, np.real(out_D["B1p_SB2_I_TF"]), label="B1p_SB2_I")
ax[4].plot(out_D.x1, np.real(out_D["B1p_SB2_Q_TF"]), label="B1p_SB2_Q")
ax[4].plot(out_D.x1, np.real(out_D["B1p_DC"] / 4), label="B1p_DC", linestyle="--")
ax[4].set_ylabel("DARM", fontsize=20)
ax[4].grid()
ax[4].legend()

plt.show()
fig.set_tight_layout(True)


In [ ]:
# idx0 = round(len(out_M.x1) / 2)

# OG_d = (
#     np.abs(out_D["B1p_SB2_I_TF"][idx0 + 3]) - np.abs(out_D["B1p_SB2_I_TF"][idx0 - 3])
# ) / (out_D.x1[idx0 + 3] - out_D.x1[idx0 - 3])

# OG_c = (
#     np.abs(out_C["B4_SB1_I_TF"][idx0 + 1]) - np.abs(out_C["B4_SB1_I_TF"][idx0 - 1])
# ) / (out_C.x1[idx0 + 1] - out_C.x1[idx0 - 1])

# OG_m = (
#     np.abs(out_M["B4_SB2_I_TF"][idx0 + 3]) - np.abs(out_M["B4_SB2_I_TF"][idx0 - 3])
# ) / (out_M.x1[idx0 + 3] - out_M.x1[idx0 - 3])

# OG_p = (
#     np.abs(out_P["B2_SB1_I_TF"][idx0 + 3]) - np.abs(out_P["B2_SB1_I_TF"][idx0 - 3])
# ) / (out_P.x1[idx0 + 3] - out_P.x1[idx0 - 3])

# OG_s = (
#     np.abs(out_S["B4_SB2_Q_TF"][idx0 + 3]) - np.abs(out_S["B4_SB2_Q_TF"][idx0 - 3])
# ) / (out_S.x1[idx0 + 3] - out_S.x1[idx0 - 3])

Timo: Optical gain is the slope of the error signal at 0. To calculate should not take
absolute values on both sides as done above. I corrected this below

In [ ]:
idx0 = round(len(out_M.x1) / 2)

OG_d = (
    np.real(out_D["B1p_SB2_I"][idx0 + 1]) - np.real(out_D["B1p_SB2_I"][idx0 - 1])
) / (out_D.x1[idx0 + 1] - out_D.x1[idx0 - 1])

OG_c = (np.real(out_C["B4_SB1_I"][idx0 + 1]) - np.real(out_C["B4_SB1_I"][idx0 - 1])) / (
    out_C.x1[idx0 + 1] - out_C.x1[idx0 - 1]
)

OG_m = (np.real(out_M["B4_SB2_I"][idx0 + 1]) - np.real(out_M["B4_SB2_I"][idx0 - 1])) / (
    out_M.x1[idx0 + 1] - out_M.x1[idx0 - 1]
)

OG_p = (np.real(out_P["B2_SB1_I"][idx0 + 1]) - np.real(out_P["B2_SB1_I"][idx0 - 1])) / (
    out_P.x1[idx0 + 1] - out_P.x1[idx0 - 1]
)

OG_s = (np.real(out_S["B4_SB2_Q"][idx0 + 1]) - np.real(out_S["B4_SB2_Q"][idx0 - 1])) / (
    out_S.x1[idx0 + 1] - out_S.x1[idx0 - 1]
)

In [ ]:
print(
    OG_d * 180 / 1064e-9,
    OG_c * 180 / 1064e-9,
    OG_m * 180 / 1064e-9,
    OG_p * 180 / 1064e-9,
    OG_s * 180 / 1064e-9,
)

## Sensing Matrix: check TFs

In [ ]:
out = {}
dofs = ["PRCL", "MICH", "SRCL", "DARM", "CARM"]

for dof in dofs:
    kat = matrix_kat.deepcopy()
    kat.parse(f"""
    fsig(1) 
    sgen test1 {dof}.AC
    xaxis(fsig.f , log, 1e-3, 1e5, 1000)
    """)

    out[dof] = kat.run()


In [ ]:
fig, ax = plt.subplots(5, 1, figsize=(8, 16))

ax[0].loglog(out["PRCL"].x1, abs(out["PRCL"]["B2_SB1_I_TF"]), label="B2_SB1_I")
ax[0].loglog(out["PRCL"].x1, abs(out["PRCL"]["B2_SB1_Q_TF"]), label="B2_SB1_Q")
ax[0].set_ylabel("PRCL", fontsize=20)
ax[0].grid()
ax[0].legend()

ax[1].loglog(out["MICH"].x1, abs(out["MICH"]["B4_SB2_I_TF"]), label="B4_SB2_I")
ax[1].loglog(out["MICH"].x1, abs(out["MICH"]["B4_SB2_Q_TF"]), label="B4_SB2_Q")
ax[1].set_ylabel("MICH", fontsize=20)
ax[1].grid()
ax[1].legend()

ax[2].loglog(out["SRCL"].x1, abs(out["SRCL"]["B4_SB2_I_TF"]), label="B4_SB2_I")
ax[2].loglog(out["SRCL"].x1, abs(out["SRCL"]["B4_SB2_Q_TF"]), label="B4_SB2_Q")
ax[2].set_ylabel("SRCL", fontsize=20)
ax[2].grid()
ax[2].legend()

ax[3].loglog(out["CARM"].x1, abs(out["CARM"]["B4_SB1_I_TF"]), label="B4_SB1_I")
ax[3].loglog(out["CARM"].x1, abs(out["CARM"]["B4_SB1_Q_TF"]), label="B4_SB1_Q")
ax[3].set_ylabel("CARM", fontsize=20)
ax[3].grid()
ax[3].legend()

ax[4].loglog(out["DARM"].x1, abs(out["DARM"]["B1p_SB2_I_TF"]), label="B1p_SB2_I")
ax[4].loglog(out["DARM"].x1, abs(out["DARM"]["B1p_SB2_Q_TF"]), label="B1p_SB2_Q")
ax[4].set_ylabel("DARM", fontsize=20)
ax[4].grid()
ax[4].legend()

plt.show()
fig.set_tight_layout(True)

## Sensing Matrix: computation

In [ ]:
M_row = {}

sensors = ["B1p_SB2_I_TF", "B4_SB1_I_TF", "B4_SB2_I_TF", "B2_SB1_I_TF", "B4_SB2_Q_TF"]

for sensor in sensors:
    if sensor != "B4_SB1_I_TF":
        M_row[sensor] = [
            abs(out["DARM"][sensor][1])
            * np.sign(np.cos(np.angle(out["DARM"][sensor][0])))
            / out_PDs[sensor.split("_")[0] + "_DC"],
            abs(out["CARM"][sensor][1])
            * np.sign(np.cos(np.angle(out["CARM"][sensor][0])))
            * 1e-6
            / out_PDs[sensor.split("_")[0] + "_DC"],
            abs(out["MICH"][sensor][1])
            * np.sign(np.cos(np.angle(out["MICH"][sensor][0])))
            / out_PDs[sensor.split("_")[0] + "_DC"],
            abs(out["PRCL"][sensor][1])
            * np.sign(np.cos(np.angle(out["PRCL"][sensor][0])))
            / out_PDs[sensor.split("_")[0] + "_DC"],
            abs(out["SRCL"][sensor][1])
            * np.sign(np.cos(np.angle(out["SRCL"][sensor][0])))
            / out_PDs[sensor.split("_")[0] + "_DC"],
        ]
    else:
        M_row[sensor] = [
            abs(out["DARM"][sensor][1])
            * np.sign(np.cos(np.angle(out["DARM"][sensor][0])))
            / out_PDs[sensor.split("_")[0] + "_DC"],
            abs(out["CARM"][sensor][1])
            * np.sign(np.cos(np.angle(out["CARM"][sensor][0])))
            / out_PDs[sensor.split("_")[0] + "_DC"],
            abs(out["MICH"][sensor][1])
            * np.sign(np.cos(np.angle(out["MICH"][sensor][0])))
            / out_PDs[sensor.split("_")[0] + "_DC"],
            abs(out["PRCL"][sensor][1])
            * np.sign(np.cos(np.angle(out["PRCL"][sensor][0])))
            / out_PDs[sensor.split("_")[0] + "_DC"],
            abs(out["SRCL"][sensor][1])
            * np.sign(np.cos(np.angle(out["SRCL"][sensor][0])))
            / out_PDs[sensor.split("_")[0] + "_DC"],
        ]

    # M_row[sensor] = abs(M_row[sensor] / np.linalg.norm(M_row[sensor]))

In [ ]:
mydata = [
    ["B1p_SB2_I", *M_row["B1p_SB2_I_TF"]],
    ["B4_SB1_I", *M_row["B4_SB1_I_TF"]],
    ["B4_SB2_I", *M_row["B4_SB2_I_TF"]],
    ["B2_SB1_I", *M_row["B2_SB1_I_TF"]],
    ["B4_SB2_Q", *M_row["B4_SB2_Q_TF"]],
]
# create header
head = ["DARM [W/m]", "CARM*1e-6 [W/m]", "MICH [W/m]", "PRCL [W/m]", "SRCL [W/m]"]
# display table
print(f"LSC Sensing Matrix (DC) ---- RAW!")
print(tabulate(mydata, headers=head, tablefmt="grid"))

## Prepare the lock

Timo: the gain of the error signals should note be scaled by the DC power. Also in the
calculation here the readouts with unoptimized phases are used instead of the optimized
readouts. Some error signal signs are wrong.

In [ ]:
#               err        dof           -1 / err_OG (from SM) * W/deg -> W/m  *  gain_factor       tolerance
lock = f"""
lock DARM_lock B1p_SB2_I  DARM.DC {-1 / (M_row["B1p_SB2_I_TF"][0] * 1064e-9 / 180) * 10}     5e-14         enabled=True offset=0
lock CARM_lock B4_SB1_I   CARM.DC {-1 / (M_row["B4_SB1_I_TF"][1] * 1064e-9 / 180) * 1e-3}     5e-14         enabled=True offset=0
lock PRCL_lock B2_SB1_I   PRCL.DC {-1 / (M_row["B2_SB1_I_TF"][3] * 1064e-9 / 180) * 0.5e-1}     1e-12          enabled=True offset=0
lock MICH_lock B4_SB2_I   MICH.DC {-1 / (M_row["B4_SB2_I_TF"][2] * 1064e-9 / 180) * 1e-4}     1e-12          enabled=True offset=0
lock SRCL_lock B4_SB2_Q   SRCL.DC {-1 / (M_row["B4_SB2_Q_TF"][4] * 1064e-9 / 180) * 1e-3}     1e-10          enabled=True offset=0
"""

In [ ]:
print(lock)

## Test the combination of TMs tilt

In [ ]:
test_kat = matrix_kat.deepcopy()
test_kat.parse("""
mathd NE NETM.xbeta
mathd NI NITM.xbeta
mathd WE WETM.xbeta
mathd WI WITM.xbeta
""")

In [ ]:
out = test_kat.run(fac.Xaxis("DIFFm_set", "lin", -1e-7, 1e-7, 300))

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 3))
ax.plot(out.x1, out["NE"], label="NE")
ax.plot(out.x1, out["NI"], label="NI")
ax.plot(out.x1, out["WE"], label="WE")
ax.plot(out.x1, out["WI"], label="WI")
ax.legend()
ax.grid()
ax.set_xlabel("var [rad]")
ax.set_ylabel("mir [rad]")
fig.set_tight_layout(True)

In [ ]:
tele = f"""
s sB2_to_bench B2_pickoff.p4 B2_split.p1 L=0
bs B2_split R=0.5 T=0.5 alpha=45
s B2_nf_tele B2_split.p2 B2_nf_port.p1 user_gouy_x={waist_gouy} user_gouy_y={waist_gouy}
s B2_ff_tele B2_split.p3 B2_ff_port.p1 user_gouy_x={waist_gouy + 90} user_gouy_y={waist_gouy + 90}
nothing B2_nf_port
nothing B2_ff_port

s sB4_to_bench PRM2.p3 B4_split.p1 L=0
bs B4_split R=0.5 T=0.5 alpha=45
s B4_nf_tele B4_split.p2 B4_nf_port.p1 user_gouy_x={waist_gouy} user_gouy_y={waist_gouy}
s B4_ff_tele B4_split.p3 B4_ff_port.p1 user_gouy_x={waist_gouy + 90} user_gouy_y={waist_gouy + 90}
nothing B4_nf_port
nothing B4_ff_port

s sB1p_to_bench B1p_pickoff.p2 B1p_split.p1 L=0
bs B1p_split R=0.5 T=0.5 alpha=45
s B1p_nf_tele B1p_split.p2 B1p_nf_port.p1 user_gouy_x={waist_gouy} user_gouy_y={waist_gouy}
s B1p_ff_tele B1p_split.p3 B1p_ff_port.p1 user_gouy_x={waist_gouy + 90} user_gouy_y={waist_gouy + 90}
nothing B1p_nf_port
nothing B1p_ff_port

s sB7_to_bench NETM.p2 B7_split.p1 L=0
bs B7_split R=0.5 T=0.5 alpha=45
s B7_nf_tele B7_split.p2 B7_nf_port.p1 user_gouy_x={waist_gouy} user_gouy_y={waist_gouy}
s B7_ff_tele B7_split.p3 B7_ff_port.p1 user_gouy_x={waist_gouy + 90} user_gouy_y={waist_gouy + 90}
nothing B7_nf_port
nothing B7_ff_port

s sB8_to_bench WETM.p2 B8_split.p1 L=0
bs B8_split R=0.5 T=0.5 alpha=45
s B8_nf_tele B8_split.p2 B8_nf_port.p1 user_gouy_x={waist_gouy} user_gouy_y={waist_gouy}
s B8_ff_tele B8_split.p3 B8_ff_port.p1 user_gouy_x={waist_gouy + 90} user_gouy_y={waist_gouy + 90}
nothing B8_nf_port
nothing B8_ff_port
"""


## Sweep all angular dofs

In [ ]:
lock_kat = matrix_kat.deepcopy()
lock_kat.modes(maxtem=4)

lock_kat.parse(lock)
lock_kat.parse(tele)
lock_kat.parse_file("QDs_DC.kat")
lock_kat.parse_file("QDs_RF.kat")

make_SoS_QDs("B1p", lock_kat, lock_kat.eom3.f - lock_kat.eom2.f)
make_SoS_QDs("B1p", lock_kat, lock_kat.eom2.f - lock_kat.eom1.f)

lock_kat.add_frequency(2 * lock_kat.eom1.f)
lock_kat.add_frequency(-2 * lock_kat.eom1.f)
lock_kat.add_frequency(2 * lock_kat.eom2.f)
lock_kat.add_frequency(-2 * lock_kat.eom2.f)

make_2f_QDs("B2", lock_kat, lock_kat.eom1.f.value)
make_2f_QDs("B2", lock_kat, lock_kat.eom2.f.value)
make_2f_QDs("B4", lock_kat, lock_kat.eom1.f.value)
make_2f_QDs("B4", lock_kat, lock_kat.eom2.f.value)
make_2f_QDs("B1p", lock_kat, lock_kat.eom1.f.value)
make_2f_QDs("B1p", lock_kat, lock_kat.eom2.f.value)

In [ ]:
print(lock)

In [ ]:
out = {}
AA_dofs = [
    "PRM1",
    "PRM2",
    "PRM3",
    "SRM1",
    "SRM2",
    "SRM3",
    "BS",
    "IBshift",
    "IBtilt",
    "DIFFp",
    "DIFFm",
    "COMMp",
    "COMMm",
]

### Try just running the first lock
Timo: the locks below run well because method='newton' is chosen which does not use
lock gains. If we use method='proportional' we see that error signal become bigger
and bigger because the sign of the gain is wrong in some cases.

In [ ]:
locktest_kat = lock_kat.deepcopy()
locktest_kat.run(fac.RunLocks(method="newton", display_progress=True))
print()
locktest_kat = lock_kat.deepcopy()
locktest_kat.run(
    fac.RunLocks(
        method="proportional",
        scale_factor=0.2,
        display_progress=True,
        max_iterations=100,
    )
)

In [ ]:
for dof in AA_dofs:
    print(f"Now sweeping {dof}...", end="\r")
    time_start = time.time()

    out[dof] = lock_kat.run(
        f"xaxis({dof + '_set'}, lin, 0, 1e-12, 3, pre_step=RunLocks(method='newton', show_progress_bar=True))"
    )

    time_stop = time.time()
    elapsed_time = time_stop - time_start
    print(f"Done sweeping {dof}. Elapsed time: {elapsed_time:.2f} seconds.")
print("All done")

In [ ]:
print(QD_RF_data_tmp)

In [ ]:
for dof in AA_dofs:
    QD_RF_filename = out_path + dof + "/" + dof + "_QD_RF.txt"
    QD_SoS_filename = out_path + dof + "/" + dof + "_QD_SoS.txt"
    QD_2f_filename = out_path + dof + "/" + dof + "_QD_2f.txt"
    QD_DC_filename = out_path + dof + "/" + dof + "_QD_DC.txt"
    PD_DC_filename = out_path + dof + "/" + dof + "_PD_DC.txt"
    pdh_filename = out_path + dof + "/" + dof + "_err.txt"
    wp_filename = out_path + dof + "/" + dof + "_wp.txt"

    QD_RF_names = out_path + "/QD_RF_names.txt"
    QD_SoS_names = out_path + "/QD_SoS_names.txt"
    QD_2f_names = out_path + "/QD_2f_names.txt"
    QD_DC_names = out_path + "/QD_DC_names.txt"
    PD_DC_names = out_path + "/PD_DC_names.txt"
    pdh_names = out_path + "/pdh_names.txt"

    QD_RF_data_tmp = []
    QD_SoS_data_tmp = []
    QD_2f_data_tmp = []
    QD_DC_data_tmp = []
    PD_DC_data_tmp = []
    pdh_data_tmp = []
    wp_data_tmp = []

    x = out[dof].x1[:, np.newaxis]

    QD_RF_names_handle = open(QD_RF_names, "w")
    QD_SoS_names_handle = open(QD_SoS_names, "w")
    QD_2f_names_handle = open(QD_2f_names, "w")
    QD_DC_names_handle = open(QD_DC_names, "w")
    PD_DC_names_handle = open(PD_DC_names, "w")
    pdh_names_handle = open(pdh_names, "w")

    for _ in out[dof].detectors:
        if len(_.split("_")) > 2 and _.split("_")[2] in ["SB1", "SB2", "SB4"]:
            ## select QDs RF
            QD_RF_data_tmp.append(out[dof][_])
            QD_RF_names_handle.write(_ + "\n")

        if len(_.split("_")) > 2 and _.split("_")[2] in [
            f"{(lock_kat.eom3.f - lock_kat.eom2.f) * 1e-6:.0f}MHz"
        ]:
            ## select QDs SoS
            QD_SoS_data_tmp.append(out[dof][_])
            QD_SoS_names_handle.write(_ + "\n")

        if len(_.split("_")) > 2 and _.split("_")[2] in [
            f"{(2 * lock_kat.eom1.f) * 1e-6:.0f}MHz",
            f"{(2 * lock_kat.eom2.f) * 1e-6:.0f}MHz",
        ]:
            ## select QDs 2f
            QD_2f_data_tmp.append(out[dof][_])
            QD_2f_names_handle.write(_ + "\n")

        if len(_.split("_")) == 3 and _.split("_")[1] in ["nf", "ff"]:
            ## select QDs DC
            QD_DC_data_tmp.append(out[dof][_])
            QD_DC_names_handle.write(_ + "\n")

        if _ in ["B2_DC", "B4_DC", "B1_DC", "B7_DC", "B8_DC"]:
            ## select PDs DC
            PD_DC_data_tmp.append(out[dof][_])
            PD_DC_names_handle.write(_ + "\n")

        if _ in ["B2_SB1_I", "B4_SB2_Q", "B4_SB2_Q", "B4_SB1_I", "B1p_SB2_I"]:
            pdh_data_tmp.append(out[dof][_])
            pdh_names_handle.write(_ + "\n")

    QD_RF_names_handle.close()
    QD_SoS_names_handle.close()
    QD_2f_names_handle.close()
    QD_DC_names_handle.close()
    PD_DC_names_handle.close()
    pdh_names_handle.close()

    QD_RF_data_handle = open(QD_RF_filename, "w+")
    np.savetxt(QD_RF_filename, np.hstack([x, np.asarray(QD_RF_data_tmp).T]))
    QD_RF_data_handle.close()

    QD_SoS_data_handle = open(QD_SoS_filename, "w+")
    np.savetxt(QD_SoS_filename, np.hstack([x, np.asarray(QD_SoS_data_tmp).T]))
    QD_SoS_data_handle.close()

    QD_2f_data_handle = open(QD_2f_filename, "w+")
    np.savetxt(QD_2f_filename, np.hstack([x, np.asarray(QD_2f_data_tmp).T]))
    QD_2f_data_handle.close()

    QD_DC_data_handle = open(QD_DC_filename, "w+")
    np.savetxt(QD_DC_filename, np.hstack([x, np.asarray(QD_DC_data_tmp).T]))
    QD_DC_data_handle.close()

    PD_DC_data_handle = open(PD_DC_filename, "w+")
    np.savetxt(PD_DC_filename, np.hstack([x, np.asarray(PD_DC_data_tmp).T]))
    PD_DC_data_handle.close()

    pdh_data_handle = open(pdh_filename, "w+")
    np.savetxt(pdh_filename, np.hstack([x, np.asarray(pdh_data_tmp).T]))
    pdh_data_handle.close()

    # wp_data_handle = open(wp_filename, "w+")
    # np.savetxt(wp_filename, np.hstack([x, np.asarray(wp_data_tmp).T]))
    # wp_data_handle.close()